# 11_phase4_normalisation_sensitivity.ipynb
Phase 4 — Normalisation method sensitivity analysis (GSE176078)

**Question:** does the choice of normalisation target (baseline: fixed target_sum=1e4) materially change downstream clustering and the headline HER2+ Memory T cell finding?

**Alternative tested:** `target_sum=None` — scanpy normalises each cell to the **median** total count across the dataset instead of a fixed arbitrary value. This is a standard, well-recognised alternative (not an arbitrary choice), avoiding the need for a brand-new normalisation package (e.g. scran, SCTransform) given today's dependency-related issues with new tools.

**Same pipeline, same seeding/memory-safe approach as the QC threshold sensitivity notebook** — only the normalisation step differs from baseline.

In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# NUMBA_NUM_THREADS set before scanpy import — required for reproducible,
# single-threaded clustering (see QC sensitivity notebook for full reasoning).
# ----------------------------
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import scrublet as scr
import matplotlib.pyplot as plt
from scipy.sparse import issparse, csr_matrix, vstack
from pathlib import Path

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_sensitivity"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_sensitivity"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# ----------------------------
# Cell 2 — Pipeline function, parameterised by normalisation target_sum
# Same validated, memory-safe pipeline as before. QC threshold fixed at
# the validated baseline (n_genes>500) — only normalisation changes here,
# isolating that one variable.
# ----------------------------
def run_pipeline_with_normalisation(target_sum, label):
    print(f"\n{'='*70}")
    print(f"RUNNING PIPELINE WITH target_sum={target_sum} ({label})")
    print(f"{'='*70}\n")

    adata = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")
    if issparse(adata.X):
        adata.X = adata.X.tocsr().astype("float32", copy=False)
    else:
        adata.X = csr_matrix(adata.X, dtype=np.float32)
    gc.collect()

    vdj_prefixes = ("IGHV", "IGLV", "IGKV", "TRAV", "TRBV", "TRGV", "TRDV",
                     "IGHD", "IGHJ", "IGLJ", "IGKJ")
    constant_genes = ("TRAC", "TRBC", "TRGC", "TRDC", "IGHA", "IGHD", "IGHE",
                       "IGHG", "IGHM", "IGLC", "IGKC")
    all_prefixes = vdj_prefixes + constant_genes
    vdj_mask = ~adata.var_names.str.startswith(all_prefixes)
    mt_mask = ~adata.var_names.str.startswith("MT-")
    adata = adata[:, vdj_mask & mt_mask].copy()
    gc.collect()

    n_cells = adata.n_obs
    n_genes_by_counts = np.zeros(n_cells, dtype=np.float32)
    chunk_size = 2000
    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)
        chunk = adata.X[start:end]
        n_genes_by_counts[start:end] = np.asarray((chunk > 0).sum(axis=1)).flatten()
    adata.obs["n_genes_by_counts"] = n_genes_by_counts

    # Fixed at the VALIDATED baseline QC threshold — only normalisation varies
    adata = adata[adata.obs["n_genes_by_counts"] > 500].copy()
    sc.pp.filter_genes(adata, min_cells=3)
    print(f"After QC filter (n_genes>500, baseline): {adata.n_obs} cells")
    gc.collect()

    all_predicted_doublets = np.zeros(adata.n_obs, dtype=bool)
    samples = adata.obs["orig.ident"].unique()
    for sample in samples:
        sample_mask = adata.obs["orig.ident"] == sample
        sample_idx = np.where(sample_mask)[0]
        if len(sample_idx) < 50:
            continue
        chunks = []
        for start in range(0, len(sample_idx), chunk_size):
            end = min(start + chunk_size, len(sample_idx))
            global_idx = sample_idx[start:end]
            chunk = adata.X[global_idx]
            if not issparse(chunk): chunk = csr_matrix(chunk)
            chunks.append(chunk)
        X_sample = vstack(chunks)
        try:
            scrub = scr.Scrublet(X_sample)
            _, predicted_doublets = scrub.scrub_doublets(verbose=False)
            all_predicted_doublets[sample_idx] = predicted_doublets
        except Exception as e:
            print(f"  Scrublet failed for {sample}: {e}")
        gc.collect()

    adata.obs["predicted_doublet"] = all_predicted_doublets
    adata = adata[~adata.obs["predicted_doublet"]].copy()
    print(f"After doublet removal: {adata.n_obs} cells")
    gc.collect()

    # ---- The variable under test ----
    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)
    adata.raw = adata

    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
    adata = adata[:, adata.var.highly_variable].copy()
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack", random_state=0)
    sce.pp.harmony_integrate(adata, key="orig.ident", basis="X_pca", random_state=0)
    gc.collect()

    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)
    sc.tl.leiden(adata, resolution=0.6, key_added="leiden_0.6",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
    n_clusters = adata.obs["leiden_0.6"].nunique()
    print(f"\nFinal: {adata.n_obs} cells, {n_clusters} clusters at resolution 0.6")

    return adata, n_clusters

print("Pipeline function ready")

Pipeline function ready


In [3]:
# ----------------------------
# Cell 3 — Run with target_sum=None (median-based normalisation)
# vs baseline (target_sum=1e4, fixed value)
# ----------------------------
adata_median_norm, n_clusters_median = run_pipeline_with_normalisation(
    None, "MEDIAN-based (alternative to fixed 1e4)"
)
gc.collect()


RUNNING PIPELINE WITH target_sum=None (MEDIAN-based (alternative to fixed 1e4))

After QC filter (n_genes>500, baseline): 91622 cells
After doublet removal: 91425 cells


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-14 15:56:50,001 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-14 15:57:38,564 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-14 15:57:39,515 - harmonypy - INFO - Iteration 1 of 10
2026-07-14 15:58:56,426 - harmonypy - INFO - Iteration 2 of 10
2026-07-14 16:00:10,428 - harmonypy - INFO - Iteration 3 of 10
2026-07-14 16:01:29,003 - harmonypy - INFO - Iteration 4 of 10
2026-07-14 16:02:48,700 - harmonypy - INFO - Iteration 5 of 10
2026-07-14 16:04:08,298 - harmonypy - INFO - Converged after 5 iterations
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Final: 91425 cells, 25 clusters at resolution 0.6


80824

In [4]:
# ----------------------------
# Cell 4 — Check headline finding + cluster count vs baseline
# Baseline (target_sum=1e4, validated): 91,425 cells, 28 clusters,
# HER2+ Memory T cells credibly elevated vs both ER+ and TNBC (scCODA)
# ----------------------------
def check_memory_t_pattern(adata, label):
    adata_raw = adata.raw.to_adata()
    for gene in ["CD3D", "IL7R", "CCR7"]:
        if gene not in adata_raw.var_names:
            print(f"  {label}: {gene} not found, skipping check")
            return None
    cd3d = adata_raw[:, "CD3D"].X.toarray().flatten()
    il7r = adata_raw[:, "IL7R"].X.toarray().flatten()
    ccr7 = adata_raw[:, "CCR7"].X.toarray().flatten()
    is_memory_t_like = (cd3d > 0.5) & (il7r > 1.0) & (ccr7 > 0.3)
    adata.obs["memory_t_like"] = is_memory_t_like
    if "subtype" not in adata.obs.columns:
        print(f"  {label}: subtype metadata not found, skipping")
        return None
    prop_by_subtype = adata.obs.groupby("subtype", observed=True)["memory_t_like"].mean() * 100
    print(f"  {label}: memory-T-like cell %% by subtype:")
    print(f"    {prop_by_subtype.to_dict()}")
    return prop_by_subtype

prop_median = check_memory_t_pattern(adata_median_norm, "Median-based normalisation")

comparison = pd.DataFrame({
    "Normalisation": ["Fixed (target_sum=1e4, baseline)", "Median-based (target_sum=None)"],
    "Cells": [91425, adata_median_norm.n_obs],
    "Clusters (res 0.6)": [28, n_clusters_median],
})
print("\n" + comparison.to_string(index=False))
comparison.to_csv(RESULTS_DIR / "GSE176078_normalisation_sensitivity_summary.csv", index=False)
if prop_median is not None:
    prop_median.to_csv(RESULTS_DIR / "GSE176078_median_norm_memoryT_check.csv")

print("\n>>> Baseline (target_sum=1e4): Memory T cells credibly elevated in HER2+")
print(">>> vs both ER+ and TNBC. Check above: does HER2+ still show the highest")
print(">>> memory-T-like %% under median-based normalisation too?")

  Median-based normalisation: memory-T-like cell %% by subtype:
    {'ER+': 2.89103481163567, 'HER2+': 6.632541929862774, 'TNBC': 4.626793895061885}

                   Normalisation  Cells  Clusters (res 0.6)
Fixed (target_sum=1e4, baseline)  91425                  28
  Median-based (target_sum=None)  91425                  25

>>> Baseline (target_sum=1e4): Memory T cells credibly elevated in HER2+
>>> vs both ER+ and TNBC. Check above: does HER2+ still show the highest
>>> memory-T-like %% under median-based normalisation too?
